# RNAbpFlow — Google Colab Inference Notebook

**Project:** RNA 3D Structure Prediction for Template-Sparse RNA Sequences  
**Model:** RNAbpFlow  
**Official repository:** `https://github.com/Bhattacharya-Lab/RNAbpFlow`

RNAbpFlow uses **RNA sequence + base-pair map(s)**. Do not use the experimental native 3D structure as an inference input. For the primary benchmark, base-pair maps must come from one fixed, documented prediction pipeline.

## 1. GPU check

Enable a GPU in Colab: **Runtime → Change runtime type → GPU**.

In [ ]:
import os, sys, platform, subprocess, time, threading
from pathlib import Path

print("Host Python:", sys.version)
print("Platform:", platform.platform())
subprocess.run(["nvidia-smi"], check=False)

## 2. Clone the official RNAbpFlow repository

In [ ]:
%cd /content
if not os.path.exists("/content/RNAbpFlow"):
    !git clone https://github.com/Bhattacharya-Lab/RNAbpFlow.git
else:
    print("RNAbpFlow repository already exists; reusing it.")

%cd /content/RNAbpFlow
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("RNAbpFlow commit:", commit)

## 3. Install Micromamba

An isolated environment is used because the official RNAbpFlow environment pins Python 3.10, PyTorch 2.1.2 and CUDA 11.8.

In [ ]:
%cd /content
if not os.path.exists("/content/bin/micromamba"):
    !curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj
!/content/bin/micromamba --version

## 4. Create the official RNAbpFlow environment

In [ ]:
%cd /content/RNAbpFlow
env_check = subprocess.run(
    ["/content/bin/micromamba", "-r", "/content/micromamba_root", "env", "list"],
    capture_output=True, text=True
)
if "RNAbpFlow" in env_check.stdout:
    print("RNAbpFlow environment already exists.")
else:
    !MAMBA_ROOT_PREFIX=/content/micromamba_root /content/bin/micromamba create -y -n RNAbpFlow -f RNAbpFlow.yml

## 5. Apply verified Colab compatibility fixes

The verified 2026 Colab runtime required:
- `mkl<2024.1` to avoid `libtorch_cpu.so: undefined symbol: iJIT_NotifyEvent`
- `numpy<2` for PyTorch 2.1.2 compatibility

In [ ]:
!/content/bin/micromamba -r /content/micromamba_root install \
  -n RNAbpFlow \
  -y \
  -c conda-forge \
  "mkl<2024.1" \
  "numpy<2"

## 6. Verify the isolated environment

The verification should return code 0 and show Python 3.10, NumPy 1.x, PyTorch 2.1.2, CUDA=True and the GPU name.

In [ ]:
verify_code = """
import sys, numpy, torch
print("Python:", sys.version)
print("NumPy:", numpy.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
"""

result = subprocess.run(
    ["/content/bin/micromamba", "-r", "/content/micromamba_root",
     "run", "-n", "RNAbpFlow", "python", "-c", verify_code],
    capture_output=True, text=True
)
print("RETURN CODE:", result.returncode)
print("\nSTDOUT:\n", result.stdout)
print("\nSTDERR:\n", result.stderr)
if result.returncode != 0:
    raise RuntimeError("RNAbpFlow environment verification failed.")

## 7. Download and place the trained checkpoint

The default inference configuration expects `checkpoint/RNA3DB.ckpt`.

In [ ]:
%cd /content/RNAbpFlow
CHECKPOINT_ARCHIVE = "/content/Checkpoints.tar.gz"

if not os.path.exists(CHECKPOINT_ARCHIVE):
    !wget -q --show-progress -O /content/Checkpoints.tar.gz "https://zenodo.org/records/19388910/files/Checkpoints.tar.gz?download=1"

if not os.path.exists("/content/rnabpflow_checkpoints_extracted"):
    !mkdir -p /content/rnabpflow_checkpoints_extracted
    !tar -xzf /content/Checkpoints.tar.gz -C /content/rnabpflow_checkpoints_extracted

import shutil
checkpoint_dir = Path("/content/RNAbpFlow/checkpoint")
checkpoint_dir.mkdir(exist_ok=True)
matches = list(Path("/content/rnabpflow_checkpoints_extracted").rglob("RNA3DB.ckpt"))
if not matches:
    raise FileNotFoundError("RNA3DB.ckpt not found.")
dest_ckpt = checkpoint_dir / "RNA3DB.ckpt"
if not dest_ckpt.exists():
    shutil.copy2(matches[0], dest_ckpt)

print("Checkpoint ready:", dest_ckpt)
print("Size (MB):", round(dest_ckpt.stat().st_size / (1024**2), 2))

## 8. Inspect bundled example inputs

Each target contains a FASTA file and three base-pair maps. A single map may be replicated three times if only one map is available.

In [ ]:
%cd /content/RNAbpFlow
!find Inputs -maxdepth 2 -type f | sort
print("\nCurrent list.txt:")
!cat Inputs/list.txt

## 9. Optional lightweight verification target

Use one bundled target and one sample for a quick smoke test. Skip this cell if you want to run the original full example list.

In [ ]:
from pathlib import Path
import shutil

list_file = Path("/content/RNAbpFlow/Inputs/list.txt")
backup_file = Path("/content/RNAbpFlow/Inputs/list.txt.backup")
if not backup_file.exists():
    shutil.copy2(list_file, backup_file)

TEST_TARGET = "2oiu_P"
TEST_SAMPLES = 1
list_file.write_text(f"{TEST_TARGET} {TEST_SAMPLES}\n")
print(list_file.read_text())

## 10. Run RNAbpFlow inference + runtime / GPU-memory logging

In [ ]:
%cd /content/RNAbpFlow

gpu_samples = []
stop_monitor = False

def monitor_gpu():
    global stop_monitor
    while not stop_monitor:
        try:
            out = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                text=True
            ).strip().splitlines()
            if out:
                gpu_samples.append(int(out[0]))
        except Exception:
            pass
        time.sleep(0.5)

monitor_thread = threading.Thread(target=monitor_gpu, daemon=True)
monitor_thread.start()
start = time.perf_counter()

proc = subprocess.run(
    ["/content/bin/micromamba", "-r", "/content/micromamba_root",
     "run", "-n", "RNAbpFlow", "python", "inference.py"],
    cwd="/content/RNAbpFlow",
    capture_output=True,
    text=True
)

elapsed = time.perf_counter() - start
stop_monitor = True
monitor_thread.join(timeout=2)
peak_gpu_mb = max(gpu_samples) if gpu_samples else None

print("\n===== STDOUT =====")
print(proc.stdout)
print("\n===== STDERR =====")
print(proc.stderr)
print("\n=== RUN SUMMARY ===")
print("Exit code:", proc.returncode)
print(f"Runtime: {elapsed:.2f} s")
print("Peak observed GPU memory:", f"{peak_gpu_mb} MiB" if peak_gpu_mb is not None else "Unavailable")

if proc.returncode != 0:
    raise RuntimeError("RNAbpFlow inference failed. Check STDERR above.")

## 11. Inspect generated predictions

In [ ]:
prediction_root = Path("/content/RNAbpFlow/Predictions")
if not prediction_root.exists():
    raise FileNotFoundError("Predictions directory was not created.")

files = sorted([p for p in prediction_root.rglob("*") if p.is_file()])
print(f"Generated files: {len(files)}")
for p in files[:100]:
    print(p)

pdb_files = sorted(prediction_root.rglob("*.pdb"))
cif_files = sorted(prediction_root.rglob("*.cif"))
print("\nPDB files:", len(pdb_files))
print("CIF files:", len(cif_files))
if not pdb_files and not cif_files:
    raise RuntimeError("No structure files were generated.")

## 12. Save reproducibility metadata

In [ ]:
import csv

gpu_name = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    text=True
).strip().splitlines()[0]

version_code = (
    "import sys,numpy,torch;"
    "print(sys.version.split()[0]);"
    "print(numpy.__version__);"
    "print(torch.__version__)"
)
versions = subprocess.check_output(
    ["/content/bin/micromamba", "-r", "/content/micromamba_root",
     "run", "-n", "RNAbpFlow", "python", "-c", version_code],
    text=True
).strip().splitlines()

record = {
    "model": "RNAbpFlow",
    "model_commit": commit,
    "checkpoint": "RNA3DB.ckpt",
    "python": versions[0],
    "numpy": versions[1],
    "pytorch": versions[2],
    "runtime_seconds": round(elapsed, 3),
    "gpu": gpu_name,
    "peak_gpu_memory_mib_observed": peak_gpu_mb,
    "output_root": str(prediction_root),
    "exit_code": proc.returncode
}

LOG_PATH = "/content/rnabpflow_run_log.csv"
with open(LOG_PATH, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=record.keys())
    writer.writeheader()
    writer.writerow(record)

print("Saved:", LOG_PATH)
print(record)

## 13. Optional — save outputs to Google Drive

Large checkpoints and prediction folders should not be committed to GitHub.

In [ ]:
# OPTIONAL CELL
# from google.colab import drive
# import shutil, os
# drive.mount("/content/drive")
# destination = "/content/drive/MyDrive/RNA3D_Benchmark/RNAbpFlow"
# os.makedirs(destination, exist_ok=True)
# shutil.copytree("/content/RNAbpFlow/Predictions",
#                 os.path.join(destination, "Predictions"),
#                 dirs_exist_ok=True)
# shutil.copy2(LOG_PATH, destination)
# print("Saved to:", destination)

## 14. Future benchmark workflow

For each benchmark target prepare:

```text
Inputs/TARGET_ID/
├── TARGET_ID.fasta
├── map1.npy
├── map2.npy
└── map3.npy
```

Add the target and desired sample count to `Inputs/list.txt`.

**Primary benchmark rule:** base-pair maps must come from a fixed predicted-BP pipeline. Native/ground-truth base pairs may only be used in a separately labelled oracle experiment.